Notebook for exploring creating this project's Transform function. Don't forget to set the Kernel to this project's .venv

Code for transforming stock price json into csv for Snowflake ingestion

In [ ]:
import json
from datetime import UTC, datetime
from io import StringIO

import boto3
import pandas as pd


def daily_stock_price(data):
    stockprice_list = []
    for row in data:
        date = row['date']
        symbol = row['symbol']
        open_price = row['open']
        high_price = row['high']
        low_price = row['low']
        close_price = row['close']
        vwap = row['vwap']
        volume = row['volume']
        price_change = row['change']
        price_change_percent = row['changePercent']
        stockprice_list.append({
            'date': date,
            'symbol': symbol,
            'open_price': open_price,
            'high_price': high_price,
            'low_price': low_price,
            'close_price': close_price,
            'vwap': vwap,
            'volume': volume,
            'price_change': price_change,
            'price_change_percent': price_change_percent
        })
    return stockprice_list
    
s3 = boto3.client("s3")
Bucket = "phile-findata1-raw-data"
Key = "to_process/"
all_stock_data = []

# List all files waiting to be processed
response = s3.list_objects(Bucket=Bucket, Prefix=Key)
stockprice_files = [file['Key'] for file in response['Contents'] if 'stock_prices' in file['Key']]


for file_key in stockprice_files:
    # Read raw JSON from S3
    response = s3.get_object(Bucket=Bucket, Key=file_key)
    content = response['Body'].read().decode('utf-8')
    data = json.loads(content)
    all_stock_data.extend(data)

    # Use Transform function
stockprice_list = daily_stock_price(all_stock_data)

# Create DataFrames & dedup
stockprice_df = pd.DataFrame(stockprice_list).drop_duplicates(subset=['date', 'symbol'])

# Convert dates
stockprice_df['date'] = pd.to_datetime(stockprice_df['date'], format='%Y-%m-%d')

# Write transformed data to S3 as CSV
timestamp = datetime.now(UTC).strftime("%Y%m%d_%H%M%S")

for df, name in [(stockprice_df, "daily_stock_prices")]:
    buffer = StringIO()
    df.to_csv(buffer, index=False)
    s3.put_object(
        Bucket="phile-findata1-transformed-data",
        Key=f"{name}/{name}_transformed_{timestamp}.csv",
        Body=buffer.getvalue()
    )

Code to transform Customer profile data to build out dims in Snowflake

In [11]:
import json
from datetime import UTC, datetime
from io import StringIO

import boto3
import pandas as pd


def company_profiles(data):
    profile_list = []
    for row in data:
        symbol = row['symbol']
        company_name = row['companyName']
        cik = row['cik']
        isin = row['isin']
        cusip = row['cusip']
        sector = row['sector']
        industry = row['industry']
        ipo_date = row['ipoDate']
        exchange_code = row['exchange']
        exchange_full_name = row['exchangeFullName']
        currency = row['currency']
        is_actively_trading = row['isActivelyTrading']
        is_etf = row['isEtf']
        is_adr = row['isAdr']
        is_fund = row['isFund']
        profile_list.append({
            'symbol': symbol,
            'company_name': company_name,
            'cik': cik,
            'isin': isin,
            'cusip': cusip,
            'sector': sector,
            'industry': industry,
            'ipo_date': ipo_date,
            'exchange_code': exchange_code,
            'exchange_full_name': exchange_full_name,
            'currency': currency,
            'is_actively_trading': is_actively_trading,
            'is_etf': is_etf,
            'is_adr': is_adr,
            'is_fund': is_fund
        })
    return profile_list
    
s3 = boto3.client("s3")
Bucket = "phile-findata1-raw-data"
Key = "to_process/"

# List all files waiting to be processed
response = s3.list_objects(Bucket=Bucket, Prefix=Key)
company_profile_files = [file['Key'] for file in response['Contents'] if 'company_profile' in file['Key']]

for file_key in company_profile_files:
    # Read raw JSON from S3
    response = s3.get_object(Bucket=Bucket, Key=file_key)
    content = response['Body'].read().decode('utf-8')
    data = json.loads(content)

    # Use Transform function
    profile_list = company_profiles(data)

    # Create DataFrames & dedup
    companyprofile_df = pd.DataFrame(profile_list).drop_duplicates(subset=['symbol'])

    # Convert dates
    companyprofile_df['ipo_date'] = pd.to_datetime(companyprofile_df['ipo_date'], format='%Y-%m-%d')

    # Write transformed data to S3 as CSV
    timestamp = datetime.now(UTC).strftime("%Y%m%d_%H%M%S")

    for df, name in [(companyprofile_df, "company_profiles")]:
        buffer = StringIO()
        df.to_csv(buffer, index=False)
        s3.put_object(
            Bucket="phile-findata1-transformed-data",
            Key=f"{name}/{name}_transformed_{timestamp}.csv",
            Body=buffer.getvalue().encode("utf-8"),
            ContentType="text/csv",
            ContentDisposition="inline"
        )